# DSP Exercise Set: Signals & Time-Domain Analysis

## Procesamiento Digital de Señales

### Facultad de ingenieria

Profesor: Oscar Francisco Fuentes Casarrubias

In [ ]:
# === Datos del estudiante ===
NOMBRE = ["Hernández González Andrés Sebastian - 319328439", "Luna Reyes Javier - 319046351"]
GRUPO = 4

for nom in NOMBRE:
  print(f"Nombre: {nom}\n")
print("Grupo:", GRUPO)

Nombre: Hernández González Andrés Sebastian - 319328439

Nombre: Luna Reyes Javier - 319046351

Grupo: 4


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from ipywidgets import interact, IntSlider, widgets
from IPython.display import display

# DATOS FIJOS
N      = 8
X_COLS = [1.0, 2.5, 4.0, 5.5, 6.8]
Y_ROWS = [7.5, 6.5, 5.5, 4.5, 3.5, 2.5, 1.5, 0.5]

INPUT_LABELS  = ['x(0)','x(4)','x(2)','x(6)','x(1)','x(5)','x(3)','x(7)']
OUTPUT_LABELS = ['X(0)','X(1)','X(2)','X(3)','X(4)','X(5)','X(6)','X(7)']

BUTTERFLIES = {
    1: [(0,1),(2,3),(4,5),(6,7)],
    2: [(0,2),(1,3),(4,6),(5,7)],
    3: [(0,4),(1,5),(2,6),(3,7)],
}

TWIDDLE = {
    1: {(0,1):'W⁰',(2,3):'W⁰',(4,5):'W⁰',(6,7):'W⁰'},
    2: {(0,2):'W⁰',(1,3):'W²',(4,6):'W⁰',(5,7):'W²'},
    3: {(0,4):'W⁰',(1,5):'W¹',(2,6):'W²',(3,7):'W³'},
}

# Colores
BG       = '#12121E'
FG       = '#E8E8F0'
DIM_LINE = '#2E2E55'
DIM_NODE = '#2E2E55'
C_INPUT  = '#AAAACC'
C_DONE   = '#7ED348'
STAGE_COLOR = {
    1: '#4FA3E8',   # azul
    2: '#2ECC8F',   # verde
    3: '#F07030',   # naranja
}

# Secuencia de pasos
all_steps = [('entrada', None, None)]
for stage in [1, 2, 3]:
    for bf in BUTTERFLIES[stage]:
        all_steps.append((stage, bf[0], bf[1]))
all_steps.append(('done', None, None))

STEP_NAMES = []
for s in all_steps:
    if s[0] == 'entrada':
        STEP_NAMES.append('0 — Entrada (bit-reversal)')
    elif s[0] == 'done':
        STEP_NAMES.append(f'{len(all_steps)-1} — FFT Completa ✓')
    else:
        stage, ra, rb = s
        tw = TWIDDLE[stage].get((ra, rb), 'W⁰')
        idx = all_steps.index(s)
        STEP_NAMES.append(f'{idx} — Etapa {stage}: filas {ra}↔{rb}  {tw}')

# Función de dibujo

def draw_fft(paso=0):
    fig, ax = plt.subplots(figsize=(14, 9))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.set_xlim(0, 8)
    ax.set_ylim(-1.2, 9.1)
    ax.set_aspect('equal')
    ax.axis('off')

    step = all_steps[paso]

    # Determinar qué etapas ya están "completadas"
    completed_stages = set()
    active_stage = None
    if step[0] not in ('entrada', 'done'):
        active_stage = step[0]
        for i in range(1, paso):
            s = all_steps[i]
            if s[0] not in ('entrada', 'done'):
                completed_stages.add(s[0])
        completed_stages.discard(active_stage)

    if step[0] == 'done':
        completed_stages = {1, 2, 3}

    #Título
    ax.set_title('FFT de 8 puntos — Diagrama de Mariposas (DIT Radix-2)',
                 fontsize=15, fontweight='bold', color=FG, pad=14)

    #Separadores
    for x in [1.75, 3.25, 4.75, 6.15]:
        ax.axvline(x=x, color='#2A2A44', lw=0.9, ls='--', zorder=1)

    #Encabezados de columna
    col_titles = ['Entrada\n(bit-reversal)', 'Etapa 1\ndist = 1',
                  'Etapa 2\ndist = 2',       'Etapa 3\ndist = 4', 'Salida']
    for i, (txt, x) in enumerate(zip(col_titles, X_COLS)):
        if i == 0 or i == 4:
            c = FG
        else:
            c = STAGE_COLOR[i]
        ax.text(x, 8.65, txt, ha='center', va='bottom',
                fontsize=9, color=c, fontweight='bold')

    #Etiquetas entrada / salida
    for r in range(N):
        y = Y_ROWS[r]
        ax.text(X_COLS[0] - 0.6, y, INPUT_LABELS[r],
                ha='right', va='center', fontsize=10.5, color=FG)
        ax.text(X_COLS[4] + 0.12, y, OUTPUT_LABELS[r],
                ha='left', va='center', fontsize=10.5, color=FG)

    #Helper: flecha
    def arr(x0, y0, x1, y1, color, lw=1.0, alpha=1.0, z=2):
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color=color,
                                   lw=lw, alpha=alpha, mutation_scale=10),
                    zorder=z)

    def circ(x, y, color, r=0.11, z=4):
        ax.add_patch(plt.Circle((x, y), r, color=color, zorder=z))

    # Dibujar todas las conexiones de fondo
    for stage in [1, 2, 3]:
        ci = stage - 1
        x0, x1 = X_COLS[ci], X_COLS[ci + 1]

        if stage in completed_stages:
            line_color = STAGE_COLOR[stage]
            line_lw    = 1.2
            line_alpha = 0.55
        elif stage == active_stage:
            line_color = DIM_LINE
            line_lw    = 0.9
            line_alpha = 1.0
        else:
            line_color = DIM_LINE
            line_lw    = 0.9
            line_alpha = 1.0

        for (ra, rb) in BUTTERFLIES[stage]:
            ya, yb = Y_ROWS[ra], Y_ROWS[rb]
            for ys_, ye in [(ya,ya),(ya,yb),(yb,ya),(yb,yb)]:
                arr(x0, ys_, x1, ye, line_color,
                    lw=line_lw, alpha=line_alpha)

    # Líneas de salida
    out_color = C_DONE if step[0] == 'done' else DIM_LINE
    for r in range(N):
        arr(X_COLS[3], Y_ROWS[r], X_COLS[4], Y_ROWS[r],
            out_color, lw=1.5 if step[0]=='done' else 0.9)

    # Nodos intermedios
    for ci in [1, 2, 3]:
        st = ci  # etapa correspondiente
        for r in range(N):
            if st in completed_stages:
                nc = STAGE_COLOR[st]
                nr = 0.13
            else:
                nc = DIM_NODE
                nr = 0.11
            circ(X_COLS[ci], Y_ROWS[r], nc, r=nr, z=4)

    # Nodos entrada
    cin = C_INPUT if step[0] in ('entrada',) else '#444460'
    if step[0] == 'done':
        cin = C_INPUT
    for r in range(N):
        circ(X_COLS[0], Y_ROWS[r], cin, r=0.10, z=4)

    #MARIPOSA ACTIVA
    if step[0] not in ('entrada', 'done'):
        stage, ra, rb = step
        color = STAGE_COLOR[stage]
        tw    = TWIDDLE[stage].get((ra, rb), 'W⁰')

        ci = stage - 1
        x0, x1 = X_COLS[ci], X_COLS[ci + 1]
        ya, yb  = Y_ROWS[ra], Y_ROWS[rb]

        # 4 flechas activas
        for ys_, ye in [(ya,ya),(ya,yb),(yb,ya),(yb,yb)]:
            arr(x0, ys_, x1, ye, color, lw=2.8, z=8)

        # Nodos activos
        for r_act in [ra, rb]:
            for cx in [ci, ci + 1]:
                circ(X_COLS[cx], Y_ROWS[r_act], color, r=0.17, z=7)

        # Label twiddle
        xm = (x0 + x1) / 2
        ym = (ya + yb) / 2 + 0.28
        ax.text(xm, ym, tw, ha='center', va='bottom',
                fontsize=13, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', fc=BG,
                          ec=color, lw=1.8),
                zorder=9)

        # Signo −1
        ax.text(x1 - 0.1, yb + 0.18, '−1',
                fontsize=9, color=color, fontweight='bold',
                ha='center', zorder=9)

    #todos los nodos verdes
    if step[0] == 'done':
        for ci in [1, 2, 3]:
            for r in range(N):
                circ(X_COLS[ci], Y_ROWS[r], C_DONE, r=0.15, z=7)
        for r in range(N):
            arr(X_COLS[3], Y_ROWS[r], X_COLS[4], Y_ROWS[r],
                C_DONE, lw=2.2, z=6)

    #Barra de estado
    if step[0] == 'entrada':
        msg = '⟶  Entrada en orden bit-reversal'
        ec  = C_INPUT
    elif step[0] == 'done':
        msg = '✓  FFT completa — 12 operaciones  (DFT directa: 64)'
        ec  = C_DONE
    else:
        stage, ra, rb = step
        tw  = TWIDDLE[stage].get((ra, rb), 'W⁰')
        ec  = STAGE_COLOR[stage]
        msg = (f'Etapa {stage}  ▸  Mariposa: filas {ra} ↔ {rb}  '
               f'▸  Factor de giro: {tw}  '
               f'▸  Distancia = {2**(stage-1)}')

    ax.text(4.0, -0.82, msg, ha='center', va='center',
            fontsize=10.5, color=FG,
            bbox=dict(boxstyle='round,pad=0.55', fc='#1A1A30',
                      ec=ec, lw=1.5),
            zorder=10)

    #Fórmula
    ax.text(0.05, -0.82, r'$W_N^k = e^{-j2\pi k/N}$',
            ha='left', va='center', fontsize=10, color='#9999CC')

    #Leyenda
    leg_elems = [
        Line2D([0],[0], color=STAGE_COLOR[1], lw=2.5, label='Etapa 1  dist=1'),
        Line2D([0],[0], color=STAGE_COLOR[2], lw=2.5, label='Etapa 2  dist=2'),
        Line2D([0],[0], color=STAGE_COLOR[3], lw=2.5, label='Etapa 3  dist=4'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor=C_DONE,
               markersize=9, label='Completo'),
    ]
    ax.legend(handles=leg_elems, loc='lower right', fontsize=9,
              facecolor='#1A1A30', edgecolor='#444466', labelcolor=FG)

    #Contador de paso
    ax.text(7.95, 8.65,
            f'Paso {paso}/{len(all_steps)-1}',
            ha='right', va='top', fontsize=9, color='#777799')

    plt.tight_layout()
    plt.show()

# WIDGET INTERACTIVO
slider = IntSlider(
    value=0,
    min=0,
    max=len(all_steps) - 1,
    step=1,
    description='Paso:',
    continuous_update=False,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='700px')
)

# Botones siguiente / anterior
btn_prev = widgets.Button(description='◀ Anterior',
                          layout=widgets.Layout(width='130px'))
btn_next = widgets.Button(description='Siguiente ▶',
                          layout=widgets.Layout(width='130px'))

def on_prev(b):
    if slider.value > 0:
        slider.value -= 1

def on_next(b):
    if slider.value < slider.max:
        slider.value += 1

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

controls = widgets.HBox([btn_prev, btn_next, slider])
out = widgets.interactive_output(draw_fft, {'paso': slider})

display(controls, out)

Output()